In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# CONFIG – edit these paths
# ============================================================
CSV_PATHS = [
    "../data/apple_labeled/apple_isolated_group1_6h_labeled.csv",
    "../data/apple_labeled/apple_apartment_group1_6h_1_labeled.csv",
    "../data/apple_labeled/apple_apartment_group1_6h_2_labeled.csv",
    "../data/apple_labeled/apple_apartment_group1_6h_3_labeled.csv",
    "../data/apple_labeled/apple_uni_group1_2h_labeled.csv",
]

BIN_SIZE_SEC = 10
OUTPUT_DIR = Path("feature_results")
OUTPUT_DIR.mkdir(exist_ok=True)

REPORT_FILE = OUTPUT_DIR / "full_feature_report.txt"
CORR_CSV    = OUTPUT_DIR / "activity_correlations.csv"
BURST_CSV   = OUTPUT_DIR / "burst_shape_summaries.csv"
PAYLOAD_CSV = OUTPUT_DIR / "payload_top_values.csv"

# ============================================================
# Analysis function
# ============================================================
def analyse_dataset(csv_path: str, bin_size: int = 10):
    cols = [
        "time", "source", "labelled_device", "data",
        "delta_start_start", "delta_end_start", "time_start_end", "rssi"
    ]
    df = pd.read_csv(csv_path, usecols=cols)

    results = {
        "n_packets": len(df),
        "device_counts": df["labelled_device"].value_counts().to_dict(),
    }

    # ----- 1. Payload evolution (top 10 data values per device) -----
    payload_rows = []
    for dev in ["AirPods", "iPad", "AirTag"]:
        if dev not in df["labelled_device"].values:
            continue
        vc = df.loc[df["labelled_device"] == dev, "data"].value_counts().head(10)
        for rank, (val, cnt) in enumerate(vc.items(), 1):
            payload_rows.append({
                "device": dev,
                "rank": rank,
                "data": val,
                "count": int(cnt)
            })
    results["payload"] = pd.DataFrame(payload_rows)

    # ----- 2. Activity correlation -----
    df["t_bin"] = (df["time"] // bin_size).astype(int)
    counts = (
        df.groupby(["t_bin", "labelled_device"])
          .size()
          .unstack(fill_value=0)
    )
    results["corr"] = counts.corr().round(4)
    results["n_bins"] = len(counts)

    # ----- 3. Burst shape statistics -----
    burst_cols = ["delta_start_start", "delta_end_start", "time_start_end"]
    results["burst_describe"] = (
        df.groupby("labelled_device")[burst_cols]
          .describe()
          .round(3)
    )

    # Compact summary for CSV
    agg = df.groupby("labelled_device")[burst_cols].agg(
        ["count", "mean", "std", "min",
         lambda x: x.quantile(0.25), "median",
         lambda x: x.quantile(0.75), "max", "nunique"]
    ).round(3)
    agg.columns = [f"{col}_{stat}" for col, stat in agg.columns]
    results["burst_summary"] = agg

    return results

# ============================================================
# Main loop
# ============================================================
all_corrs = []
all_bursts = []
all_payloads = []

with open(REPORT_FILE, "w", encoding="utf-8") as report:
    report.write("Full Feature Report – Activity Correlation, Burst Shape & Payload\n")
    report.write("=" * 80 + "\n\n")

    for i, path in enumerate(CSV_PATHS, 1):
        path = Path(path)
        name = path.stem

        report.write(f"\n{'='*80}\n")
        report.write(f"DATASET {i}: {name}\n")
        report.write(f"{'='*80}\n")
        report.write(f"File: {path}\n\n")

        try:
            res = analyse_dataset(str(path), bin_size=BIN_SIZE_SEC)
        except Exception as e:
            report.write(f"ERROR: {e}\n")
            print(f"[ERROR] {name}: {e}")
            continue

        # --- header info ---
        report.write(f"Total packets : {res['n_packets']:,}\n")
        report.write(f"Time bins     : {res['n_bins']:,}  ({BIN_SIZE_SEC}s each)\n")
        report.write("Device counts :\n")
        for dev, cnt in res["device_counts"].items():
            report.write(f"  {dev:10s}: {cnt:,}\n")
        report.write("\n")

        # --- 1. Payload ---
        report.write("-" * 40 + "\n1. PAYLOAD EVOLUTION (top 10 data values)\n" + "-" * 40 + "\n")
        if not res["payload"].empty:
            for dev in res["payload"]["device"].unique():
                report.write(f"\n{dev}:\n")
                sub = res["payload"][res["payload"]["device"] == dev]
                for _, row in sub.iterrows():
                    report.write(f"  {row['count']:6d}  {row['data']}\n")
        report.write("\n")

        # --- 2. Correlation ---
        report.write("-" * 40 + "\n2. ACTIVITY CORRELATION\n" + "-" * 40 + "\n")
        report.write(res["corr"].to_string())
        report.write("\n\n")

        # --- 3. Burst shape ---
        report.write("-" * 40 + "\n3. BURST SHAPE STATISTICS\n" + "-" * 40 + "\n")
        report.write(res["burst_describe"].to_string())
        report.write("\n\n")

        # --- collect for CSVs (FIXED flattening) ---
        corr_flat = (
            res["corr"]
            .stack()
            .rename_axis(index=["device_A", "device_B"])
            .reset_index(name="correlation")
        )
        corr_flat.insert(0, "dataset", name)
        all_corrs.append(corr_flat)

        burst_flat = res["burst_summary"].reset_index()
        burst_flat.insert(0, "dataset", name)
        all_bursts.append(burst_flat)

        payload_flat = res["payload"].copy()
        payload_flat.insert(0, "dataset", name)
        all_payloads.append(payload_flat)

        print(f"[OK] {name} – {res['n_packets']:,} packets")

# ============================================================
# Save CSV tables
# ============================================================
if all_corrs:
    pd.concat(all_corrs, ignore_index=True).to_csv(CORR_CSV, index=False)
    print(f"Saved → {CORR_CSV}")

if all_bursts:
    pd.concat(all_bursts, ignore_index=True).to_csv(BURST_CSV, index=False)
    print(f"Saved → {BURST_CSV}")

if all_payloads:
    pd.concat(all_payloads, ignore_index=True).to_csv(PAYLOAD_CSV, index=False)
    print(f"Saved → {PAYLOAD_CSV}")

print(f"\nFull text report → {REPORT_FILE}")
print("Done.")

[OK] apple_isolated_group1_6h_labeled – 508,284 packets
[OK] apple_apartment_group1_6h_1_labeled – 837,635 packets
[OK] apple_apartment_group1_6h_2_labeled – 577,060 packets
[OK] apple_apartment_group1_6h_3_labeled – 972,920 packets
[OK] apple_uni_group1_2h_labeled – 2,530,814 packets
Saved → feature_results\activity_correlations.csv
Saved → feature_results\burst_shape_summaries.csv
Saved → feature_results\payload_top_values.csv

Full text report → feature_results\full_feature_report.txt
Done.
